# NYC Property Sales - Data Cleaning

This notebook cleans the NYC Citywide Annualized Calendar Sales dataset for analysis.

The cleaning process includes:

- Removing exact duplicate records
- Removing the completely blank EASE-MENT field
- Standardizing numeric and date fields
- Handling missing values in core analytical fields
- Retaining valid records with missing optional fields
- Removing unnecessary geographic and identifier fields
- Validating the final cleaned dataset
- Exporting the cleaned dataset for exploratory analysis

In [1]:
import pandas as pd
import os

df = pd.read_csv("../data/raw/nyc_sales_property_raw.csv")

print(f"Raw dataset shape: {df.shape}")
df.head()

Raw dataset shape: (845607, 29)


C:\Users\capul\AppData\Local\Temp\ipykernel_33648\2483262908.py:4: DtypeWarning: Columns (11: RESIDENTIAL UNITS, 12: COMMERCIAL UNITS, 13: TOTAL UNITS) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/nyc_sales_property_raw.csv")


,BOROUGH,NEIGHBORHOOD,BUILDING CLASS CATEGORY,TAX CLASS AS OF FINAL ROLL,BLOCK,LOT,EASE-MENT,BUILDING CLASS AS OF FINAL ROLL,ADDRESS,APARTMENT NUMBER,...,SALE PRICE,SALE DATE,Latitude,Longitude,Community Board,Council District,BIN,BBL,Census Tract 2020,Neighborhood Tabulation Area (NTA) (2020)
0,1,CHINATOWN,47 CONDO NON-BUSINESS STORAGE,4,172,1490,NaN,RS,"50 FRANKLIN STREET, S9B",S9B,...,"$1,590,000",06/21/2019,40.716951,-74.002500,101.0,1.0,1088222.0,1.001728e+09,31.0,MN0102
1,1,EAST VILLAGE,10 COOPS - ELEVATOR APARTMENTS,2,461,24,NaN,D4,"232 EAST 6TH STREET, 6D",NaN,...,"$515,000",06/11/2019,40.727669,-73.989201,103.0,2.0,1006622.0,1.004610e+09,38.0,MN0303
2,1,FASHION,21 OFFICE BUILDINGS,4,779,23,NaN,O6,227 WEST 29TH STREET,NaN,...,$0,05/03/2019,40.748263,-73.993857,105.0,3.0,1014284.0,1.007790e+09,95.0,MN0501
3,1,FASHION,13 CONDOS - ELEVATOR APARTMENTS,2,841,1044,NaN,R4,"16 WEST 40TH STREET, 26A",26A,...,"$5,203,258",03/11/2019,40.752505,-73.982574,105.0,4.0,1090404.0,1.008418e+09,84.0,MN0502
4,1,EAST VILLAGE,10 COOPS - ELEVATOR APARTMENTS,2,431,31,NaN,D4,"141 EAST THIRD STREET, 2E",NaN,...,"$975,000",12/27/2019,40.723793,-73.985602,103.0,2.0,1005762.0,1.004310e+09,32.0,MN0303


In [24]:
print("RAW df BEFORE df_clean:")
print(df["SALE PRICE"].head(10))
print("dtype:", df["SALE PRICE"].dtype)
print("missing:", df["SALE PRICE"].isna().sum())

RAW df BEFORE df_clean:
0     $1,590,000
1       $515,000
2             $0
3     $5,203,258
4       $975,000
5     $6,900,000
6       $990,000
7    $99,350,000
8    $14,000,000
9             $0
Name: SALE PRICE, dtype: str
dtype: str
missing: 0


In [25]:
df_clean = df.copy()

print("SALE PRICE immediately after copy:")
print(df_clean["SALE PRICE"].head(10))
print("Missing:", df_clean["SALE PRICE"].isna().sum())

duplicate_count = df_clean.duplicated().sum()

print(f"\nExact duplicate rows: {duplicate_count}")

df_clean = df_clean.drop_duplicates().copy()

print(f"Shape after removing exact duplicates: {df_clean.shape}")
print(f"Rows removed: {len(df) - len(df_clean)}")

print("\nSALE PRICE after duplicate removal:")
print(df_clean["SALE PRICE"].head(10))
print("Missing:", df_clean["SALE PRICE"].isna().sum())

SALE PRICE immediately after copy:
0     $1,590,000
1       $515,000
2             $0
3     $5,203,258
4       $975,000
5     $6,900,000
6       $990,000
7    $99,350,000
8    $14,000,000
9             $0
Name: SALE PRICE, dtype: str
Missing: 0

Exact duplicate rows: 3351
Shape after removing exact duplicates: (842256, 29)
Rows removed: 3351

SALE PRICE after duplicate removal:
0     $1,590,000
1       $515,000
2             $0
3     $5,203,258
4       $975,000
5     $6,900,000
6       $990,000
7    $99,350,000
8    $14,000,000
9             $0
Name: SALE PRICE, dtype: str
Missing: 0


In [26]:
df_clean = df_clean.drop(columns=["EASE-MENT"])

print(f"Shape after removing EASE-MENT: {df_clean.shape}")
print("EASE-MENT still present:", "EASE-MENT" in df_clean.columns)

Shape after removing EASE-MENT: (842256, 28)
EASE-MENT still present: False


In [27]:
numeric_fields = [
    "RESIDENTIAL UNITS",
    "COMMERCIAL UNITS",
    "TOTAL UNITS",
    "LAND SQUARE FEET",
    "GROSS SQUARE FEET",
    "YEAR BUILT"
]

for column in numeric_fields:
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )

df_clean["SALE PRICE"] = (
    df_clean["SALE PRICE"].astype(str).str.replace("$","",regex=False).str.replace(",","", regex=False).str.strip()
)

df_clean["SALE PRICE"] = pd.to_numeric(
    df_clean["SALE PRICE"],
    errors="coerce"
)

df_clean["SALE DATE"] = pd.to_datetime(
    df_clean["SALE DATE"],
    errors="coerce"
)

df_clean.loc[
    df_clean["YEAR BUILT"] == 0,
    "YEAR BUILT"
]=pd.NA

sale_year = df_clean["SALE DATE"].dt.year

df_clean.loc[
    df_clean["YEAR BUILT"] > sale_year,
    "YEAR BUILT"
]=pd.NA

In [29]:
required_fields = [
    "TOTAL UNITS",
    "RESIDENTIAL UNITS",
    "YEAR BUILT",
    "SALE PRICE",
    "SALE DATE"
]

print("Missing values before required-field cleaning:")
print(df_clean[required_fields].isna().sum())

df_clean = df_clean.dropna(
    subset=[
        "TOTAL UNITS",
        "RESIDENTIAL UNITS",
        "YEAR BUILT"
    ]
).copy()

print("\nShape after removing missing required fields:", df_clean.shape)

print("\nMissing values after required-field cleaning:")
print(df_clean[required_fields].isna().sum())

Missing values before required-field cleaning:
TOTAL UNITS          118827
RESIDENTIAL UNITS    133126
YEAR BUILT            62930
SALE PRICE                0
SALE DATE                 0
dtype: int64

Shape after removing missing required fields: (650343, 28)

Missing values after required-field cleaning:
TOTAL UNITS          0
RESIDENTIAL UNITS    0
YEAR BUILT           0
SALE PRICE           0
SALE DATE            0
dtype: int64


In [6]:
fields_to_review = [
    "RESIDENTIAL UNITS",
    "COMMERCIAL UNITS",
    "TOTAL UNITS",
    "LAND SQUARE FEET",
    "GROSS SQUARE FEET",
    "YEAR BUILT",
    "SALE PRICE",
    "SALE DATE",
    "ZIP CODE",
    "TAX CLASS AS OF FINAL ROLL",
    "BUILDING CLASS AS OF FINAL ROLL",
    "APARTMENT NUMBER"
]

missing_summary = df_clean[fields_to_review].isna().sum()

print(f"Remaining missing values: {missing_summary}")

Remaining missing values: RESIDENTIAL UNITS                       0
COMMERCIAL UNITS                    75305
TOTAL UNITS                             0
LAND SQUARE FEET                   488312
GROSS SQUARE FEET                  472267
YEAR BUILT                              0
SALE PRICE                         650343
SALE DATE                               0
ZIP CODE                                1
TAX CLASS AS OF FINAL ROLL            225
BUILDING CLASS AS OF FINAL ROLL       225
APARTMENT NUMBER                   506008
dtype: int64


In [7]:
zero_sale_price_count = (df_clean["SALE PRICE"]==0).sum()

print(f"Records with $0 sale price: {zero_sale_price_count}")

Records with $0 sale price: 0


## Cleaning Decisions

### Exact duplicates
Exact duplicate records were removed because they represent identical rows across all available fields.

### EASE-MENT
The `EASE-MENT` field was removed because the column contained no usable values.

### TOTAL UNITS
Records with missing `TOTAL UNITS` were removed. Explicit `0` values were retained because blank values cannot be safely interpreted as zero.

### RESIDENTIAL UNITS
Records with missing `RESIDENTIAL UNITS` were removed. Blank values were not converted to zero because explicit `0` values exist and the NYC data dictionary does not define blank values as zero.

### YEAR BUILT
Records with `YEAR BUILT = 0` were treated as missing because `0` is not a valid construction year.

Records where `YEAR BUILT` occurs after the sale year were also treated as missing because they represent an invalid or inconsistent construction year.

After these values were standardized as missing, records with missing `YEAR BUILT` were removed because this field is required for property-age analysis.

Valid historical years, including very old construction years, were retained.

### SALE PRICE
`$0` sale prices were retained. According to NYC documentation, a $0 sale price indicates a transfer of ownership without cash consideration. These records will therefore be treated separately from cash-based transaction-value analysis.

### Optional fields
Missing values in fields such as `COMMERCIAL UNITS`, `LAND SQUARE FEET`, `GROSS SQUARE FEET`, `ZIP CODE`, `TAX CLASS AS OF FINAL ROLL`, `BUILDING CLASS AS OF FINAL ROLL`, and `APARTMENT NUMBER` were retained where appropriate because these fields are not required for all core analyses.

### APARTMENT NUMBER
Blank apartment numbers were retained because an apartment number may not apply to every property record.

### Geographic and identifier fields
The following fields were removed because they are not required for the core business questions:

- Latitude
- Longitude
- Community Board
- Council District
- BIN
- BBL
- Census Tract 2020
- Neighborhood Tabulation Area (NTA) (2020)

In [8]:
columns_to_remove = [
    "Latitude",
    "Longitude",
    "Community Board",
    "Council District",
    "BIN",
    "BBL",
    "Census Tract 2020",
    "Neighborhood Tabulation Area (NTA) (2020)"
]

df_clean = df_clean.drop(
    columns=columns_to_remove
).copy()

print("Final dataset shape:", df_clean.shape)

print("\nRemaining columns:")
print(df_clean.columns.tolist())

Final dataset shape: (650343, 20)

Remaining columns:
['BOROUGH', 'NEIGHBORHOOD', 'BUILDING CLASS CATEGORY', 'TAX CLASS AS OF FINAL ROLL', 'BLOCK', 'LOT', 'BUILDING CLASS AS OF FINAL ROLL', 'ADDRESS', 'APARTMENT NUMBER', 'ZIP CODE', 'RESIDENTIAL UNITS', 'COMMERCIAL UNITS', 'TOTAL UNITS', 'LAND SQUARE FEET', 'GROSS SQUARE FEET', 'YEAR BUILT', 'TAX CLASS AT TIME OF SALE', 'BUILDING CLASS AT TIME OF SALE', 'SALE PRICE', 'SALE DATE']


In [9]:
print("Final dataset validation")

print(f"Rows: {len(df_clean)}")
print(f"Columns: {len(df_clean.columns)}")
print(f"Exact duplicates: {df_clean.duplicated().sum()}")

print("\nData types:")
print(df_clean.dtypes)

print("\nMissing values in core fields:")
print(
    df_clean[
        [
            "TOTAL UNITS",
            "RESIDENTIAL UNITS",
            "YEAR BUILT",
            "SALE PRICE",
            "SALE DATE"
        ]
    ].isna().sum()
)

Final dataset validation
Rows: 650343
Columns: 20
Exact duplicates: 2962

Data types:
BOROUGH                                     int64
NEIGHBORHOOD                                  str
BUILDING CLASS CATEGORY                       str
TAX CLASS AS OF FINAL ROLL                    str
BLOCK                                       int64
LOT                                         int64
BUILDING CLASS AS OF FINAL ROLL               str
ADDRESS                                       str
APARTMENT NUMBER                              str
ZIP CODE                                  float64
RESIDENTIAL UNITS                         float64
COMMERCIAL UNITS                          float64
TOTAL UNITS                               float64
LAND SQUARE FEET                          float64
GROSS SQUARE FEET                         float64
YEAR BUILT                                float64
TAX CLASS AT TIME OF SALE                   int64
BUILDING CLASS AT TIME OF SALE                str
SALE PRICE    

In [10]:
duplicates_final = df_clean[
    df_clean.duplicated(keep=False)
]

print(f"Records involved in final duplicate groups: {len(duplicates_final)}")

duplicates_final.sort_values(["ADDRESS", "SALE DATE", "SALE PRICE"]).head(20)

Records involved in final duplicate groups: 4891


,BOROUGH,NEIGHBORHOOD,BUILDING CLASS CATEGORY,TAX CLASS AS OF FINAL ROLL,BLOCK,LOT,BUILDING CLASS AS OF FINAL ROLL,ADDRESS,APARTMENT NUMBER,ZIP CODE,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS,LAND SQUARE FEET,GROSS SQUARE FEET,YEAR BUILT,TAX CLASS AT TIME OF SALE,BUILDING CLASS AT TIME OF SALE,SALE PRICE,SALE DATE
461456,3,GOWANUS,22 STORE BUILDINGS,4,1007,172,K6,1 12TH ST EXTENSION,NaN,11215.0,0.0,6.0,6.0,NaN,NaN,1968.0,4,K6,NaN,2018-05-23
461817,3,GOWANUS,22 STORE BUILDINGS,4,1007,172,K6,1 12TH ST EXTENSION,NaN,11215.0,0.0,6.0,6.0,NaN,NaN,1968.0,4,K6,NaN,2018-05-23
281776,4,BAYSIDE,13 CONDOS - ELEVATOR APARTMENTS,2,5893,1404,R4,1 BAY CLUB DRIVE,19R,11360.0,1.0,0.0,1.0,0.0,0.0,1972.0,2,R4,NaN,2016-05-06
282117,4,BAYSIDE,13 CONDOS - ELEVATOR APARTMENTS,2,5893,1404,R4,1 BAY CLUB DRIVE,19R,11360.0,1.0,0.0,1.0,0.0,0.0,1972.0,2,R4,NaN,2016-05-06
252078,1,GREENWICH VILLAGE-WEST,13 CONDOS - ELEVATOR APARTMENTS,2,602,1122,R4,1 MORTON SQUARE,14A E,10014.0,1.0,0.0,1.0,0.0,0.0,2002.0,2,R4,NaN,2016-01-07
252233,1,GREENWICH VILLAGE-WEST,13 CONDOS - ELEVATOR APARTMENTS,2,602,1122,R4,1 MORTON SQUARE,14A E,10014.0,1.0,0.0,1.0,0.0,0.0,2002.0,2,R4,NaN,2016-01-07
480360,1,MURRAY HILL,21 OFFICE BUILDINGS,4,862,67,O6,10 EAST 33RD STREET,NaN,10016.0,0.0,13.0,13.0,NaN,NaN,1908.0,4,O6,NaN,2018-08-09
480935,1,MURRAY HILL,21 OFFICE BUILDINGS,4,862,67,O6,10 EAST 33RD STREET,NaN,10016.0,0.0,13.0,13.0,NaN,NaN,1908.0,4,O6,NaN,2018-08-09
486911,1,UPPER WEST SIDE (59-79),10 COOPS - ELEVATOR APARTMENTS,2,1118,22,D4,"10 WEST 66TH STREET, 32A",NaN,10023.0,0.0,0.0,0.0,0.0,0.0,1969.0,2,D4,NaN,2018-09-06
487242,1,UPPER WEST SIDE (59-79),10 COOPS - ELEVATOR APARTMENTS,2,1118,22,D4,"10 WEST 66TH STREET, 32A",NaN,10023.0,0.0,0.0,0.0,0.0,0.0,1969.0,2,D4,NaN,2018-09-06


## Final Cleaned Dataset

The cleaned dataset is prepared for exploratory analysis and business-focused analysis of NYC property sales.

The final dataset:
- Removes exact duplicate records from the original dataset
- Removes the completely blank `EASE-MENT` field
- Standardizes numeric and date fields
- Removes records with missing values in required analytical fields
- Retains valid records with missing optional fields
- Retains $0 ownership-transfer records
- Removes unnecessary geographic and identifier fields
- Preserves the core fields needed to analyze sales activity, transaction values, property characteristics, and location patterns

In [11]:
output_path = "../data/processed/nyc_sales_property_cleaned.csv"

df_clean.to_csv(output_path, index=False)

print(f"Saved cleaned dataset to: {output_path}")
print(f"Final shape: {df_clean.shape}")
print(f"File exists: {os.path.exists(output_path)}")

Saved cleaned dataset to: ../data/processed/nyc_sales_property_cleaned.csv
Final shape: (650343, 20)
File exists: True


In [12]:
df_final = pd.read_csv("../data/processed/nyc_sales_property_cleaned.csv")

df_final["SALE DATE"] = pd.to_datetime(df_final["SALE DATE"], errors="coerce")

core_fields = [
    "TOTAL UNITS",
    "RESIDENTIAL UNITS",
    "YEAR BUILT",
    "SALE PRICE",
    "SALE DATE"
]

print(f"Shape: {df_final.shape}")

print("\nCore field missing values:")
print(df_final[core_fields].isna().sum())

print("\nExact duplicate rows:")
print(df_final.duplicated().sum())

print("\n$0 sale price records:")
print((df_final["SALE PRICE"] == 0).sum())

print("\nRequired columns present:")
print(all(column in df_final.columns for column in core_fields))

print("\nData types:")
print(df_final[core_fields].dtypes)

Shape: (650343, 20)

Core field missing values:
TOTAL UNITS               0
RESIDENTIAL UNITS         0
YEAR BUILT                0
SALE PRICE           650343
SALE DATE                 0
dtype: int64

Exact duplicate rows:
2962

$0 sale price records:
0

Required columns present:
True

Data types:
TOTAL UNITS                 float64
RESIDENTIAL UNITS           float64
YEAR BUILT                  float64
SALE PRICE                  float64
SALE DATE            datetime64[us]
dtype: object
